# African Bat Specimen Records with Geolocation and Taxonomic Information Exploration with `mlcroissant`
This notebook demonstrates how to browse, process, and analyze the FAIR⁲ African Bat Specimen dataset, using the [`mlcroissant`](https://mlcommons.github.io/croissant/api/python/) library.

### Dataset Source
This dataset is described via a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.h6k3-2s1g/fair2.json`

> African Bat Specimen Records with Geolocation and Taxonomic Information


In [ ]:
# Install mlcroissant if not already present
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Croissant schema URL for the FAIR^2 African Bat dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.h6k3-2s1g/fair2.json"

# Load dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print summary
print(metadata.name + ': ' + (metadata.description or ''))
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below we display the identified record sets and their corresponding fields from the metadata structure. All entities are referenced by their canonical `@id`.

In [ ]:
# Explore available record sets
all_record_sets = []
if hasattr(metadata, 'record_set'):
    # Newer mlcroissant versions use record_set (or record_sets)
    if isinstance(metadata.record_set, list):
        all_record_sets = metadata.record_set
    elif metadata.record_set is not None:
        all_record_sets = [metadata.record_set]
elif hasattr(metadata, 'recordSets'):
    all_record_sets = metadata.recordSets
else:
    print("No record sets found in metadata.")

# If record_sets field is empty, infer from the distribution if possible (for this dataset)
if not all_record_sets and hasattr(metadata, 'distribution'):
    print("Attempting to enumerate possible record sets based on distributions...")
    for d in getattr(metadata, 'distribution', []):
        print("Distribution ID:", getattr(d, '@id', getattr(d, 'id', str(d))))

# In this dataset, the main record set is implied by the single large tabular file
# According to the FAIR² Croissant file: the main record set @id is 'https://sen.science/doi/10.71728/senscience.h6k3-2s1g/records/MainRecords' (commonly)

# We'll try to programmatically list the record sets (if present)
if all_record_sets:
    print(f"Available Record Sets (@id): { [r['@id'] if isinstance(r,dict) else getattr(r,'@id',r) for r in all_record_sets] }\n")
    # Show fields for each record set
    for record_set in all_record_sets:
        rset = record_set if isinstance(record_set, dict) else record_set.to_json() if hasattr(record_set, 'to_json') else record_set
        rset_id = rset.get('@id', getattr(record_set, '@id', str(record_set)))
        print(f"Record Set: {rset_id}")
        # Try to list fields
        field_list = []
        if 'field' in rset:
            field_list = rset['field']
        elif hasattr(record_set, 'field'):
            field_list = getattr(record_set, 'field') or []
        print("  Fields:")
        for f in field_list:
            fid = f.get('@id', getattr(f,'@id',str(f))) if isinstance(f,dict) else getattr(f,'@id',str(f))
            print("   -", fid)
else:
    print("Attempting to sample the first few records of the main table...")
    # Try to programmatically guess the @id for the main record set
    # Often: main record set @id might be croissant_url.replace('.json', '/records/MainRecords')
    inferred_record_set_id = croissant_url.rstrip('.json') + '/records/MainRecords'
    try:
        for idx, rec in enumerate(dataset.records(record_set=inferred_record_set_id)):
            print(json.dumps(rec, indent=2))
            if idx > 2:
                break
        print(f"\nAssumed main record set: {inferred_record_set_id}")
    except Exception as e:
        print(f"Could not read main record set ({inferred_record_set_id}), please consult the schema.")

## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis. We reference the main record set using its `@id`, and list available columns.

In [ ]:
# We'll use the main record set @id (update this to any other as needed):
main_record_set_id = croissant_url.rstrip('.json') + '/records/MainRecords'

# Optionally, you can print the field (column) names programmatically
sample_records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(sample_records)
print(f"Columns in {main_record_set_id}:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filter by year, normalize longitude or latitude, group by country or family, etc.

In [ ]:
# Explore the distribution for two numeric fields: Latitude and Longitude (by @id)
# For demonstration, let's filter specimens collected after 2000, normalize latitude, and group by Family

# Field IDs (from Croissant schema):
year_field = 'https://sen.science/doi/10.71728/senscience.h6k3-2s1g/fields/year'
lat_field = 'https://sen.science/doi/10.71728/senscience.h6k3-2s1g/fields/decimalLatitude'
family_field = 'https://sen.science/doi/10.71728/senscience.h6k3-2s1g/fields/family'

# Ensure fields exist (may differ slightly in your version, list columns if unsure)
for c in [year_field, lat_field, family_field]:
    if c not in df.columns:
        print(f"Warning: Field {c} not found in columns.")

# Convert year field to numeric for filtering
df[year_field] = pd.to_numeric(df[year_field], errors='coerce')

threshold_year = 2000
filtered_df = df[df[year_field] > threshold_year].copy()
print(f"Filtered records with {year_field} > {threshold_year}: {len(filtered_df)} records")
filtered_df[lat_field] = pd.to_numeric(filtered_df[lat_field], errors='coerce')
filtered_df = filtered_df[filtered_df[lat_field].notnull()]
filtered_df[lat_field + '_normalized'] = (filtered_df[lat_field] - filtered_df[lat_field].mean()) / filtered_df[lat_field].std()
print(filtered_df[[lat_field, lat_field + '_normalized']].head())

# Group by family
if family_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(family_field)[lat_field].mean().reset_index(name='mean_latitude')
    print(grouped_df.head())

## 5. Visualization
Visualize specimen latitudes by family for those collected after 2000. Also plot longitude distribution.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Strip to only families with at least 20 records for display
fam_counts = filtered_df[family_field].value_counts()
top_families = fam_counts[fam_counts > 20].index.tolist()
filtered_top_df = filtered_df[filtered_df[family_field].isin(top_families)]

# Plot boxplot of latitude by family
plt.figure(figsize=(10,5))
sns.boxplot(y=family_field, x=lat_field, data=filtered_top_df, showfliers=False)
plt.title('Specimen Latitude by Family (Year > 2000)')
plt.xlabel('Latitude')
plt.ylabel('Bat Family')
plt.tight_layout()
plt.show()

# Plot longitude histogram
long_field = 'https://sen.science/doi/10.71728/senscience.h6k3-2s1g/fields/decimalLongitude'
if long_field in filtered_df.columns:
    plt.figure(figsize=(8,4))
    filtered_df[long_field] = pd.to_numeric(filtered_df[long_field], errors='coerce')
    filtered_df[long_field].dropna().hist(bins=30)
    plt.title('Specimen Longitude Distribution (Year > 2000)')
    plt.xlabel('Longitude')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, you learned how to use the `mlcroissant` package to load a rich, FAIR²-compliant biodiversity dataset, referencing all structure via canonical `@id` for robust data discovery:

- Explored the dataset metadata and main table structure.
- Programmatically loaded tabular records by record set `@id`.
- Filtered, normalized, and grouped fields using ID-based references.
- Visualized family-level specimen distributions by location and time.

This pattern generalizes to any Croissant-conformant dataset, enabling cross-dataset exploration, comparison, and ML-ready data processing, all using self-describing metadata with stable identifier references.